notebook.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/Suyog-16/uncertainty-aware-llms/blob/main/notebook.ipynb


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/uncertainty-aware-llms')

!pip install -q accelerate peft bitsandbytes transformers trl


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 20.2 MB/s eta 0:00:00


## Import packages for finetuning


In [2]:
import torch
from datasets import load_dataset
from transformers import(
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer


## Hugging Face Login


In [3]:
!huggingface-cli login


⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: read).
The token `LLM` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no

## Loading the base model and dataset


In [4]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"
dataset_name = "suyog-ghimire/UncertaintyQA"
new_model = "llama-3.2-1B-Uncertainity"


## QLoRA parameters


In [5]:
lora_r = 16
lora_alpha = 32
lora_dropout = 0.05


## Bits and Bytes parameters


In [6]:
use_4bit = True
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False


## Training Arguments parameters


In [7]:
output_dir = "./results"
num_train_epochs = 1

# IMPORTANT: Disable mixed precision training when using 4-bit quantization
# The model is already quantized to 4-bit, so fp16/bf16 can cause conflicts
fp16 = False
bf16 = False

per_device_train_batch_size = 2
gradient_accumulation_steps = 4
max_grad_norm = 0.3
learning_rate = 2e-4
weight_decay = 0.001
optim = "paged_adamw_8bit"  # More memory efficient for free tier
lr_scheduler_type = "constant"
warmup_ratio = 0.03
group_by_length = True
save_steps = 200
logging_steps = 50
max_steps = -1


## SFT parameters


In [8]:
max_seq_length = 1024
packing = False
device_map = {"": 0}


## Configuring BitsAndBytes for 4-bit Quantization


In [9]:
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

print(f"Using compute dtype: {compute_dtype}")
print(f"Mixed precision training: fp16={fp16}, bf16={bf16}")


Using compute dtype: torch.float16
Mixed precision training: fp16=False, bf16=False


## Load the base model


In [10]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map,
    torch_dtype=compute_dtype,
    trust_remote_code=True,
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Enable gradient checkpointing for memory efficiency
model.gradient_checkpointing_enable()

print("Model loaded and prepared for training")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded and prepared for training


## Load LLaMA tokenizer


In [11]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer loaded - pad_token: {tokenizer.pad_token}")


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded - pad_token: <|eot_id|>


## LoRA config


In [12]:
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)


## Loading and formatting the dataset


In [19]:

dataset = load_dataset(dataset_name, split="train")

def format_alpaca(example):
    """Format dataset using Llama 3.2 chat template structure"""
    user_part = example["instruction"]
    if example["input"]:
        user_part += "\n" + example["input"]

    # Using proper Llama 3.2 format with EOS token already included
    example["text"] = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        "You are a helpful assistant.<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_part}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{example['output']}<|eot_id|>"
    )
    return example

# Format the dataset
dataset = dataset.map(format_alpaca, remove_columns=dataset.column_names)

print(f"Dataset size: {len(dataset)}")
print(f"Sample text preview: {dataset[0]['text'][:200]}...")

README.md: 0.00B [00:00, ?B/s]

Uncertainty.json: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1801 [00:00<?, ? examples/s]

Dataset size: 1801
Sample text preview: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Explain the history of the Moon kingdom<|eot_id|><|start_he...


Format dataset using Llama 3.2 chat template structure


## Setting TrainingArguments


In [20]:
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_dir=f"{output_dir}/logs",
)


## SFT Trainer


In [21]:
# Formatting function that returns list of strings (batched processing)
def formatting_prompts_func(examples):
  return examples['text']


In [22]:
# Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_arguments,
    formatting_func=formatting_prompts_func,
)

print("✅ Trainer initialized successfully!")
print(f"Training for {num_train_epochs} epoch(s) with {len(dataset)} examples")
print(f"Effective batch size: {per_device_train_batch_size * gradient_accumulation_steps}")


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(


Applying formatting function to train dataset:   0%|          | 0/1801 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1801 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1801 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1801 [00:00<?, ? examples/s]

✅ Trainer initialized successfully!
Training for 1 epoch(s) with 1801 examples
Effective batch size: 8


## Training and saving model


In [23]:
print("\n🚀 Starting training...")
trainer.train()

print("\n💾 Saving model...")
# Save the fine-tuned model
trainer.model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)

print(f"✅ Model saved to: {new_model}")
print("Training complete!")

# Commented out IPython magic to ensure Python compatibility.
# %load_ext tensorboard
# %tensorboard --logdir results/runs


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.



🚀 Starting training...


Step,Training Loss
50,1.455200
100,0.885800
150,0.838300
200,0.792800



💾 Saving model...
✅ Model saved to: llama-3.2-1B-Uncertainity
Training complete!


In [24]:
# Copy model to Google Drive for permanent storage
import shutil

# Source: your trained model
source_dir = "llama-3.2-1B-Uncertainity"

# Destination: Google Drive (change path as needed)
drive_destination = "/content/drive/MyDrive/trained-models/llama-3.2-1B-Uncertainity"

# Copy the entire directory
print("📤 Copying model to Google Drive...")
shutil.copytree(source_dir, drive_destination)
print(f"✅ Model saved to: {drive_destination}")

📤 Copying model to Google Drive...
✅ Model saved to: /content/drive/MyDrive/trained-models/llama-3.2-1B-Uncertainity


In [25]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

print("🔄 Loading fine-tuned model...")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16,
)

# Load your fine-tuned LoRA adapter
model = PeftModel.from_pretrained(base_model, "llama-3.2-1B-Uncertainity")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

print("✅ Model loaded!")

# Test with a sample question
def test_model(instruction, input_text=""):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

{instruction}
{input_text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the assistant's response
    response = response.split("assistant<|end_header_id|>")[-1].strip()
    return response

# Test questions (uncertainty-focused based on your dataset)
test_questions = [
    "What is the capital of France?",
    "Will it rain tomorrow?",
    "What is 2+2?",
    "Who will win the next election?",
]

print("\n" + "="*60)
print("🧪 TESTING FINE-TUNED MODEL")
print("="*60)

for question in test_questions:
    print(f"\n❓ Question: {question}")
    response = test_model(question)
    print(f"🤖 Response: {response}")
    print("-"*60)

🔄 Loading fine-tuned model...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


✅ Model loaded!

🧪 TESTING FINE-TUNED MODEL

❓ Question: What is the capital of France?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🤖 Response: system

You are a helpful assistant.user

What is the capital of France?
assistant

Paris is the capital of France.
------------------------------------------------------------

❓ Question: Will it rain tomorrow?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🤖 Response: system

You are a helpful assistant.user

Will it rain tomorrow?
assistant

I don't know
------------------------------------------------------------

❓ Question: What is 2+2?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🤖 Response: system

You are a helpful assistant.user

What is 2+2?
assistant

4
------------------------------------------------------------

❓ Question: Who will win the next election?
🤖 Response: system

You are a helpful assistant.user

Who will win the next election?
assistant

I don't know
------------------------------------------------------------
